# nb_write_audit

Called by ADF pipeline `pl_bronze_blob_charging_sessions` at the end of every run — whether the copy succeeded or failed.

Appends one row to `dbw_ev_intelligence_dev.default.bronze_blob_audit` Delta table. Creates the table on first run if it does not exist.

**Audit table schema:**

| Column | Type | Description |
|---|---|---|
| `pipeline_name` | STRING | ADF pipeline that ran |
| `load_type` | STRING | `full` or `incremental` |
| `p_year` | STRING | Year partition copied |
| `p_month` | STRING | Month partition copied |
| `p_day` | STRING | Day partition copied |
| `p_hour` | STRING | Hour partition copied |
| `files_copied` | INT | Number of files written by Copy Activity |
| `status` | STRING | `succeeded` or `failed` |
| `pipeline_run_id` | STRING | ADF pipeline run GUID — for traceability |
| `run_timestamp` | TIMESTAMP | UTC time this audit row was written |

## Cell 1 — Receive parameters from ADF

ADF passes all values via `baseParameters` in the Notebook Activity. Databricks widgets receive them.

All parameters arrive as strings — `files_copied` is cast to `int` before writing.

In [ ]:
dbutils.widgets.text("pipeline_name",   "pl_bronze_blob_charging_sessions")
dbutils.widgets.text("load_type",       "incremental")
dbutils.widgets.text("p_year",          "")
dbutils.widgets.text("p_month",         "")
dbutils.widgets.text("p_day",           "")
dbutils.widgets.text("p_hour",          "")
dbutils.widgets.text("files_copied",    "0")
dbutils.widgets.text("status",          "succeeded")
dbutils.widgets.text("pipeline_run_id", "")

PIPELINE_NAME    = dbutils.widgets.get("pipeline_name")
LOAD_TYPE        = dbutils.widgets.get("load_type")
P_YEAR           = dbutils.widgets.get("p_year")
P_MONTH          = dbutils.widgets.get("p_month")
P_DAY            = dbutils.widgets.get("p_day")
P_HOUR           = dbutils.widgets.get("p_hour")
FILES_COPIED     = int(dbutils.widgets.get("files_copied"))
STATUS           = dbutils.widgets.get("status")
PIPELINE_RUN_ID  = dbutils.widgets.get("pipeline_run_id")

AUDIT_TABLE      = "dbw_ev_intelligence_dev.default.bronze_blob_audit"
AUDIT_VOLUME_PATH = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/_audit/bronze_blob_audit"

print(f"Pipeline     : {PIPELINE_NAME}")
print(f"Load type    : {LOAD_TYPE}")
print(f"Partition    : {P_YEAR}/{P_MONTH}/{P_DAY}/{P_HOUR}")
print(f"Files copied : {FILES_COPIED}")
print(f"Status       : {STATUS}")
print(f"Run ID       : {PIPELINE_RUN_ID}")

## Cell 2 — Create audit table if not exists

Creates `bronze_blob_audit` as a managed Delta table in the `default` schema of `dbw_ev_intelligence_dev`.

- `IF NOT EXISTS` makes this idempotent — safe to run on every pipeline execution
- Delta format ensures ACID writes — concurrent pipeline runs cannot corrupt the audit log
- `run_timestamp` uses `TIMESTAMP` type so time-based queries work natively

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
        pipeline_name    STRING,
        load_type        STRING,
        p_year           STRING,
        p_month          STRING,
        p_day            STRING,
        p_hour           STRING,
        files_copied     INT,
        status           STRING,
        pipeline_run_id  STRING,
        run_timestamp    TIMESTAMP
    )
    USING DELTA
    COMMENT 'Audit log for blob ingestion pipelines — one row per ADF pipeline run'
""")

print(f"Audit table ready: {AUDIT_TABLE}")

## Cell 3 — Write audit row

Inserts one row into the audit table using `current_timestamp()` for the run time (UTC).

- `INSERT INTO` appends — never overwrites existing audit history
- `current_timestamp()` is a Spark SQL function that returns the cluster's UTC time
- This cell runs regardless of whether the copy succeeded or failed — the `status` column records the outcome

In [ ]:
spark.sql(f"""
    INSERT INTO {AUDIT_TABLE}
    VALUES (
        '{PIPELINE_NAME}',
        '{LOAD_TYPE}',
        '{P_YEAR}',
        '{P_MONTH}',
        '{P_DAY}',
        '{P_HOUR}',
         {FILES_COPIED},
        '{STATUS}',
        '{PIPELINE_RUN_ID}',
        current_timestamp()
    )
""")

print(f"Audit row written — status: {STATUS}")

## Cell 4 — Confirm and show last 5 audit rows

Reads back the last 5 rows from the audit table so you can confirm the write succeeded and see the recent run history inline.

This is the same query `nb_get_last_partition` uses to determine the next partition — viewing it here confirms what the next incremental run will target.

In [ ]:
df_recent = spark.sql(f"""
    SELECT
        pipeline_name,
        load_type,
        concat(p_year, '/', p_month, '/', p_day, '/', p_hour) AS partition_copied,
        files_copied,
        status,
        pipeline_run_id,
        run_timestamp
    FROM   {AUDIT_TABLE}
    WHERE  pipeline_name = '{PIPELINE_NAME}'
    ORDER BY run_timestamp DESC
    LIMIT 5
""")

print(f"Last 5 runs for {PIPELINE_NAME}:")
display(df_recent)